# Residual Stream Probing (Advanced)

Sophisticated probing of residual representations: token identity recovery,
next-token prediction quality, positional vs content information,
and feature separability.

## Why This Matters

Residual stream stream probing advanced characterizes the shared information bus that all transformer components read from and write to. Because the residual stream carries all inter-component communication, understanding its structure is fundamental to understanding the model as a whole.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_stream_probing_advanced import (
    token_identity_recovery, next_token_prediction_quality,
    positional_information_content, residual_feature_separability,
    residual_probing_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 15, 30, 45])
print('Model ready')

## Token Identity Recovery

Can we recover which token is at each position from the residual stream?

In [ ]:
for layer in range(4):
    result = token_identity_recovery(model, tokens, layer=layer)
    print(f"  Layer {layer}: accuracy={result['accuracy']:.0%}")
    for p in result['per_position']:
        flag = '✓' if p['correct'] else '✗'
        print(f"    pos {p['position']}: actual={p['actual_token']} "
              f"pred={p['predicted_token']} sim={p['similarity']:.4f} {flag}")

## Next-Token Prediction Quality

How good is the logit lens at predicting the next token?

In [ ]:
for layer in range(4):
    result = next_token_prediction_quality(model, tokens, layer=layer)
    print(f"  Layer {layer}: accuracy={result['accuracy']:.0%} "
          f"mean_rank={result['mean_rank']:.1f}")

## Positional vs Content Information

Does the residual stream carry more positional or content information?

In [ ]:
for layer in range(4):
    result = positional_information_content(model, tokens, layer=layer)
    print(f"  Layer {layer}: content={result['content_score']:.4f} "
          f"positional={result['positional_score']:.4f} -> {result['dominant']}")
    print(f"    same-token pairs: {result['n_same_token_pairs']}, "
          f"diff-token pairs: {result['n_diff_token_pairs']}")

## Feature Separability

How separable are different token types in residual space?

In [ ]:
for layer in range(4):
    result = residual_feature_separability(model, tokens, layer=layer)
    print(f"  Layer {layer}: separability={result['separability_score']:.4f} "
          f"(between={result['between_variance']:.4f}, "
          f"within={result['within_variance']:.4f})")

## Probing Summary Across Layers

In [ ]:
result = residual_probing_summary(model, tokens)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: identity={p['identity_accuracy']:.0%} "
          f"prediction={p['prediction_accuracy']:.0%} "
          f"rank={p['mean_prediction_rank']:.1f}")